# Session 1: numbers with gaps
**A. Real-world problem.** Type 0.1 + 0.2 into almost any programming language and it
does not give exactly 0.3. A spreadsheet, a calculator app and a missile computer all store
numbers with a limited number of binary digits, so between two stored numbers there is a gap.

**Plan.** (1) see what is really stored for 0.1, (2) measure the gap next to a number,
(3) watch the gap grow with the size of the number, (4) learn the safe way to compare.

In [ ]:
import pathlib
from decimal import Decimal

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: what is stored for 0.1?
Computers use binary. In binary, 0.1 is a repeating pattern 0.000110011001100... that never
ends, just as 1/3 = 0.3333... never ends in decimal. The computer keeps only the first
53 significant binary digits, so the stored number is very close to 0.1 but not equal.
`Decimal(x)` prints the exact value stored in the float `x`.

In [ ]:
def stored_value(x, digits=30):
    """Exact decimal value stored in the float x, cut to `digits` decimal places."""
    return format(Decimal(float(x)), f".{digits}f")


def sum_error_01_02():
    """(0.1 + 0.2) - 0.3 computed in floating point."""
    return (0.1 + 0.2) - 0.3

## Step 2: the gap next to a number
Stored numbers are not evenly spaced over the whole number line. `np.spacing(x)` is the gap
between x and the next larger stored number. In a format with m fraction bits, the numbers
between 2^k and 2^(k+1) are spaced exactly 2^(k - m) apart. So the gap doubles every time the
number doubles. The *relative* gap, gap / x, stays between 2^-m / 2 and 2^-m.

In [ ]:
def gap(x):
    """Gap between the positive float x and the next larger float (64-bit)."""
    return float(np.spacing(abs(float(x))))


def relative_gap(x):
    return gap(x) / abs(float(x))


def toy_gap(x, m):
    """Gap above x >= 1 in a toy format with m fraction bits: 2^(floor(log2 x) - m)."""
    return 2.0 ** (int(np.floor(np.log2(x))) - m)


def toy_numbers(m, lo=1.0, hi=16.0):
    """All numbers of the toy format with m fraction bits in [lo, hi)."""
    out = []
    k = int(np.floor(np.log2(lo)))
    while 2.0 ** k < hi:
        out.append(2.0 ** k * (1 + np.arange(2 ** m) / 2 ** m))
        k += 1
    v = np.concatenate(out)
    return v[(v >= lo) & (v < hi)]

## Step 3: comparing safely
Because of the gaps, `==` on computed floats is dangerous. Compare with a tolerance
that is relative to the size of the numbers.

In [ ]:
def nearly_equal(a, b, rel=1e-9, abs_tol=0.0):
    return abs(a - b) <= max(rel * max(abs(a), abs(b)), abs_tol)

## Step 4: see it
Left: the stored numbers of a tiny toy format (3 fraction bits) between 1 and 16: the ticks
spread out as the numbers grow. Right: the real 64-bit gap against magnitude on log axes.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("0.1 is stored as", stored_value(0.1))
    print("0.1 + 0.2 =", repr(0.1 + 0.2), "; (0.1 + 0.2) - 0.3 =", sum_error_01_02())
    for x in (1.0, 1000.0, 1e8, 1e16):
        print(f"gap above {x:9.1e} is {gap(x):.3e}  (relative {relative_gap(x):.2e})")
    print("2**53 + 1 == 2**53 ?", 2.0 ** 53 + 1 == 2.0 ** 53)

    fig, ax = plt.subplots(figsize=(7, 2.6))
    v = toy_numbers(3)
    ax.vlines(v, 0, 1, color="#2a6fdb")
    for k in range(4):
        ax.axvline(2 ** k, color="#d9534f", lw=1.4, ls=":")
    ax.set_xlim(0.8, 16.2)
    ax.set_yticks([])
    ax.set_xlabel("value (units); dotted lines mark powers of two")
    ax.set_title("A toy format with 3 fraction bits: gaps double with each power of two")
    fig.tight_layout()
    fig.savefig(OUT / "toy_number_line.png", dpi=120)

    xs = np.logspace(-6, 20, 200)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.loglog(xs, [gap(x) for x in xs], label="gap above x (64-bit float)")
    ax2.loglog(xs, xs * 2.0 ** -52, ls="--", label="x * 2^-52 (upper edge)")
    ax2.set_xlabel("magnitude x (units)")
    ax2.set_ylabel("gap to next float (same units as x)")
    ax2.set_title("The gap grows in proportion to the number")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "gap_vs_magnitude.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `stored_value(0.2)` and `stored_value(0.3)` and predict whether 0.2 + 0.1 == 0.3 before running it.
2. **Stretch.** Write `next_up(x)` using `np.nextafter(x, np.inf)` and check that `next_up(1.0) - 1.0` equals `gap(1.0)`; then do the same for 1e10.
3. **Explain.** Explain in your own words why the relative gap stays about the same while the gap itself keeps growing.

## Assessment
- **A. Conceptual.** Why can 0.5 be stored exactly but 0.1 cannot?
- **B. Numerical.** What is the gap just above 1000.0 in a 64-bit float? What is the relative gap? Does your answer pass a sanity check?
- **C. Graph and simulation interpretation.** In the graph view with 3 fraction bits, what is the gap for numbers in [8, 16) and how many numbers are stored there? What changes if you use 5 fraction bits?
- **D. Design and optimisation challenge.** Design: a toy sensor must store lengths from 1 to 16 units with a gap of at most 0.1 units everywhere. How many fraction bits m are needed? What is the cost of using more?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("0.1 + 0.2 is not exactly 0.3 in floating point", (0.1 + 0.2) != 0.3))
    checks.append(("(0.1 + 0.2) - 0.3 is about 5.55e-17", abs(L.sum_error_01_02() - 5.551115123125783e-17) < 1e-30))
    checks.append(("the stored 0.1 starts 0.1000000000000000055", L.stored_value(0.1).startswith("0.1000000000000000055")))
    checks.append(("worked example: gap above 1 is 2^-52 = 2.22e-16", L.gap(1.0) == 2.0 ** -52))
    checks.append(("worked example: gap above 2^53 is 2, so 2^53 + 1 == 2^53", L.gap(2.0 ** 53) == 2.0 and 2.0 ** 53 + 1 == 2.0 ** 53))
    checks.append(("gap doubles when the number doubles", L.gap(2000.0) == 2 * L.gap(1000.0)))
    r = [L.relative_gap(x) for x in np.logspace(-3, 12, 50)]
    checks.append(("relative gap stays between 2^-53 and 2^-52", min(r) >= 2.0 ** -53 and max(r) <= 2.0 ** -52))
    checks.append(("hand calculation: toy format m=3 has gap 0.5 above 4", L.toy_gap(4.0, 3) == 0.5))
    v = L.toy_numbers(3, 1.0, 16.0)
    checks.append(("toy format m=3 has 8 numbers per power of two (32 in [1, 16))", len(v) == 32 and v[0] == 1.0))
    checks.append(("toy numbers in [8, 16) are 1 apart", np.allclose(np.diff(v[v >= 8]), 1.0)))
    checks.append(("edge: nearly_equal accepts 0.1 + 0.2 vs 0.3 and rejects 0.3 vs 0.31",
                   L.nearly_equal(0.1 + 0.2, 0.3) and not L.nearly_equal(0.3, 0.31)))
    checks.append(("edge: nearly_equal(0, 1e-20) needs an absolute tolerance",
                   (not L.nearly_equal(0.0, 1e-20)) and L.nearly_equal(0.0, 1e-20, abs_tol=1e-15)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
